In [7]:
!pip install spacy networkx pyvis matplotlib sklearn
!python -m spacy download en_core_web_md

  Using cached pyvis-0.3.2-py3-none-any.whl.metadata (1.7 kB)
  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 64.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart run

In [8]:
import spacy
from spacy.tokens import Token
from collections import defaultdict

# Load medium model for better vectors & syntax
nlp = spacy.load("en_core_web_md")

# ============ Step 1: Read text ============
with open("text.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("=== Original Text ===")
print(text)
print()

# ============ Step 2: Preprocessing & Coreference (simple heuristic) ============
doc = nlp(text)

# Simple coreference resolver for pronouns
def resolve_coref(doc):
    last_named = None
    resolved = []
    for token in doc:
        if token.ent_type_ == "PERSON" or (token.pos_ == "PROPN" and token.text[0].isupper()):
            last_named = token.text
            resolved.append(token.text)
        elif token.lower_ in ["he", "she", "her", "his", "him"] and last_named:
            resolved.append(last_named)
        else:
            resolved.append(token.text)
    return nlp(" ".join(resolved))

doc = resolve_coref(doc)

# ============ Step 3: Extraction Patterns ============
triples = []

def is_high_quality(subj, rel, obj):
    """Reject trivial or meaningless triples"""
    bad_words = {"thing", "something", "anything", "life", "time", "way"}
    if not subj or not obj:
        return False
    if subj.lower() == obj.lower():
        return False
    if subj.lower() in bad_words or obj.lower() in bad_words:
        return False
    if rel.lower() in {"have", "has", "make", "show", "indicate"}:
        return False
    return True

for sent in doc.sents:
    subj = rel = obj = None

    # --- Pattern 1: normal SVO ---
    for token in sent:
        if "subj" in token.dep_:
            subj = token
        if "obj" in token.dep_:
            obj = token
        if token.dep_ == "ROOT":
            rel = token

    # --- Pattern 2: copula (“X is Y”) ---
    if not (subj and obj):
        for token in sent:
            if token.dep_ == "attr" and token.head.lemma_ == "be":
                subj = token.head.head if token.head.head.dep_ == "nsubj" else token.head.left_edge
                obj = token
                rel = token.head

    # --- Pattern 3: “X is the opposite of Y” ---
    if "opposite" in [t.lemma_ for t in sent]:
        subj = [t for t in sent if t.dep_ == "nsubj"]
        pobj = [t for t in sent if t.dep_ == "pobj"]
        if subj and pobj:
            subj, obj = subj[0], pobj[0]
            rel = nlp("be")[0]

    if subj and rel and obj and is_high_quality(subj.text, rel.lemma_, obj.text):
        triples.append((subj.text.lower(), rel.lemma_.lower(), obj.text.lower()))

# Remove duplicates
triples = list(set(triples))

print("=== Extracted Triples ===")
for s, r, o in triples:
    print(f"{s.capitalize()} --[{r}]--> {o.capitalize()}")

# ============ Step 4: Ground Truth ============
ground_truth = [
    ("alice", "be", "psychologist"),
    ("alice", "study", "traits"),
    ("alice", "research", "model"),
    ("openness", "correlate", "creativity"),
    ("conscientiousness", "associate", "productivity"),
    ("extraversion", "relate", "sociability"),
    ("introversion", "be", "opposite"),
    ("agreeableness", "link", "empathy"),
    ("neuroticism", "describe", "anxiety"),
]

# ============ Step 5: Relaxed Matching ============
relation_synonyms = {
    "be": {"is", "are", "am", "be", "was", "were"},
    "study": {"study", "studies", "studied", "research", "researches", "researched"},
    "correlate": {"correlate", "relate", "associate"},
    "associate": {"associate", "related", "connect", "linked"},
    "link": {"link", "connect"},
    "describe": {"describe", "define", "represent"},
}

def relaxed_match(pred, truth):
    ps, pr, po = pred
    ts, tr, to = truth
    s_match = ps == ts or ts in ps or ps in ts
    o_match = po == to or to in po or po in to
    r_match = pr == tr or pr in relation_synonyms.get(tr, set())
    return s_match and o_match and r_match

tp, fp, fn = 0, 0, 0
true_pos, false_pos, false_neg = [], [], []

for pred in triples:
    if any(relaxed_match(pred, t) for t in ground_truth):
        tp += 1; true_pos.append(pred)
    else:
        fp += 1; false_pos.append(pred)

for truth in ground_truth:
    if not any(relaxed_match(p, truth) for p in triples):
        fn += 1; false_neg.append(truth)

precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0

print("\n=== Evaluation ===")
print(f"True Positives: {tp}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"Precision: {precision:.2f}")
print(f"Recall:    {recall:.2f}")
print(f"F1 Score:  {f1:.2f}")

print("\n✅ TRUE POSITIVES:")
for s, r, o in true_pos:
    print(f"  {s} --[{r}]--> {o}")

print("\n❌ FALSE POSITIVES:")
for s, r, o in false_pos:
    print(f"  {s} --[{r}]--> {o}")

print("\n⚠️ FALSE NEGATIVES:")
for s, r, o in false_neg:
    print(f"  {s} --[{r}]--> {o}")


=== Original Text ===
Alice is a psychologist. She studies personality traits. Alice researches the Big Five model.

Openness correlates with creativity. Conscientiousness is associated with productivity. Extraversion relates to sociability.

Introversion is the opposite of extraversion. Agreeableness links to empathy. Neuroticism describes anxiety.

Studies show that personality traits remain stable throughout life. Research indicates these traits influence behavior.

=== Extracted Triples ===
Extraversion --[relate]--> Sociability
Introversion --[be]--> Extraversion
Alice --[be]--> Psychologist
Alice --[research]--> Model
Conscientiousness --[associate]--> Productivity
Neuroticism --[describe]--> Anxiety
Openness --[correlate]--> Creativity

=== Evaluation ===
True Positives: 6
False Positives: 1
False Negatives: 3
Precision: 0.86
Recall:    0.67
F1 Score:  0.75

✅ TRUE POSITIVES:
  extraversion --[relate]--> sociability
  alice --[be]--> psychologist
  alice --[research]--> model
  